# Phase 3 - Step 3: Vessel Segmentation Ensemble

## What is Model Ensembling?
Instead of relying on a single model, ensembling combines predictions from **multiple models** to produce a stronger, more reliable result. Think of it as a panel of expert doctors — if 2 out of 3 doctors agree they see a vessel, you trust them more than a single opinion.

## Why Ensemble These Specific Models?

| Model | Dice | Sensitivity | Specificity | Strength |
|-------|------|-------------|-------------|----------|
| **CS-Net** | 0.7984 | 0.8297 | 0.9723 | Best overall balance |
| **Attention U-Net** | 0.7981 | 0.7925 | **0.9787** | Most precise (fewest false positives) |
| **Swin-UNet** | 0.7725 | **0.8654** | 0.9581 | Catches thin vessels others miss |

Each model has a different **inductive bias**:
- **CS-Net** uses Channel + Spatial Attention → learns **which features** and **where** to look
- **Attention U-Net** uses scse attention on a pretrained ResNet34 → leverages **ImageNet features** with attention gating
- **Swin-UNet** uses Transformer self-attention → captures **long-range global context** (vessel tree connectivity)

By combining them, we get a model that is precise AND sensitive — the best of all worlds.

## Ensembling Strategies We Will Test

### Strategy 1: Simple Average (2 models)
```
Final = (CSNet + AttentionUNet) / 2
```
Equal voting power between the two best Dice models.

### Strategy 2: Weighted Average (3 models)
```
Final = 0.4 × CSNet + 0.4 × AttentionUNet + 0.2 × SwinUNet
```
CSNet and AttentionUNet dominate the vote (high precision), while SwinUNet gets a smaller vote to boost thin vessel detection.

### How It Works (Step by Step)
1. Pass the SAME test image through all models
2. Each model outputs a probability map (0.0 to 1.0 per pixel)
3. We **average** (or weighted-average) those probability maps
4. Threshold the combined map at 0.5 → final binary mask
5. Evaluate with Dice, IoU, Sensitivity, Specificity, etc.

In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
!pip install -q albumentations segmentation-models-pytorch timm



In [ ]:
# ============================================================
# Cell 2: Imports
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import json
import timm
import segmentation_models_pytorch as smp
from torch.utils.data import DataLoader
import os, sys



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Our shared modules sice we are going to provide them in github as moduls and python functions we will write the here

# 1 from src.dataset import RetinalDataset, get_train_transforms, get_val_transforms

"""
Retinal DR Detection - PyTorch Dataset & Augmentation Pipelines
================================================================
This module provides the RetinalDataset class that loads preprocessed
fundus images and their corresponding vessel/lesion masks.

It uses Albumentations for on-the-fly augmentation, ensuring that
geometric transforms (flip, rotate, distort) are applied identically
to both the image and its mask — critical for segmentation tasks.
"""

import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2


class RetinalDataset(Dataset):
    """
    PyTorch Dataset for retinal fundus images and segmentation masks.

    This dataset reads image-mask pairs from a CSV file generated by
    the preprocessing notebook (01_preprocessing.ipynb). Each row in
    the CSV contains relative paths to the processed image and its
    corresponding vessel mask.

    Args:
        csv_file (str): Path to the CSV split file (train/val/test).
        base_dir (str): Root directory of the processed dataset.
        transform (callable): Albumentations transform pipeline.
        mask_col (str): Column name in CSV pointing to the mask path.
                        Default 'vessel_path' for vessel segmentation.
    """

    def __init__(self, csv_file, base_dir, transform=None, mask_col='vessel_path'):
        self.df = pd.read_csv(csv_file)
        self.base_dir = base_dir
        self.transform = transform
        self.mask_col = mask_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Build full paths
        img_path = os.path.join(self.base_dir, row['img_path'])
        mask_path = os.path.join(self.base_dir, row[self.mask_col])

        # Read image (BGR -> RGB for Albumentations)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Read mask (grayscale, binary)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)  # Binarize to 0.0 / 1.0

        # Apply augmentations (same transform applied to BOTH image and mask)
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # Ensure mask has channel dimension: (1, H, W)
        if isinstance(mask, torch.Tensor):
            mask = mask.unsqueeze(0)
        else:
            mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask


# ============================================================
# Augmentation Pipelines
# ============================================================

def get_train_transforms(img_size=512):
    """
    Training augmentation pipeline.

    Geometric transforms are critical for retinal images because:
    - The retina can be photographed from any angle -> flips & rotations
    - Different cameras have different FOVs -> scale variations
    - Slight patient movement -> shift/distortion

    Color transforms handle:
    - Different fundus camera brands (Topcon vs Zeiss vs Canon)
    - Different illumination conditions
    """
    return A.Compose([
        A.Resize(img_size, img_size),

        # === Geometric Augmentations ===
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.0625,
            scale_limit=0.1,
            rotate_limit=45,
            border_mode=cv2.BORDER_CONSTANT,
            p=0.5
        ),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
        A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.2),

        # === Color / Illumination Augmentations ===
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(var_limit=(5, 25), p=0.2),

        # === Normalization (ImageNet stats for pretrained backbones) ===
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_val_transforms(img_size=512):
    """
    Validation/Test transform — NO augmentation, only resize + normalize.
    We must evaluate on clean, unaugmented images for fair comparison.
    """
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

# 2 from src.metrics import evaluate_full
"""
Retinal DR Detection - Medical Evaluation Metrics
===================================================
Comprehensive metrics for evaluating segmentation quality in clinical context.

In medical imaging, standard accuracy is NOT enough. A model that predicts
"no vessel everywhere" would get ~90% accuracy (because vessels are only ~10%
of the image), but would be completely useless clinically.

Key metrics explained:
- Dice Coefficient: Overlap between prediction and ground truth (harmonic mean of P & R)
- IoU (Jaccard): Intersection over Union — stricter than Dice
- Sensitivity (Recall): "Of all actual vessels, how many did we detect?"
  → Critical in medicine: missing a vessel = missing pathology
- Specificity: "Of all non-vessel pixels, how many did we correctly identify?"
  → Important to avoid false alarms
- Precision (PPV): "Of everything we predicted as vessel, how much was correct?"
- AUC-ROC: Overall discriminative ability across all thresholds
- AUC-PR: Better than AUC-ROC for imbalanced data (which vessel segmentation is)
"""

import numpy as np
import torch
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix


# ============================================================
# Pixel-Level Metrics (operate on binary masks)
# ============================================================

def dice_coefficient(pred, target, smooth=1e-6):
    """
    Dice = 2 * |P ∩ T| / (|P| + |T|)

    Ranges from 0 (no overlap) to 1 (perfect overlap).
    This is THE standard metric for medical image segmentation.

    Args:
        pred: Binary prediction tensor (B, 1, H, W)
        target: Binary ground truth tensor (B, 1, H, W)
        smooth: Smoothing factor to avoid division by zero
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    intersection = (pred_flat * target_flat).sum()
    return (2.0 * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)


def iou_score(pred, target, smooth=1e-6):
    """
    IoU (Jaccard Index) = |P ∩ T| / |P ∪ T|

    Stricter than Dice — penalizes false positives and false negatives more.
    IoU of 0.5 means the prediction overlaps only half the ground truth area.
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    intersection = (pred_flat * target_flat).sum()
    union = pred_flat.sum() + target_flat.sum() - intersection
    return (intersection + smooth) / (union + smooth)


def sensitivity(pred, target, smooth=1e-6):
    """
    Sensitivity (Recall / True Positive Rate)
    = TP / (TP + FN)

    "Of all real vessels in the image, what fraction did our model find?"
    In clinical settings this is CRITICAL — a missed vessel could mean
    missing a hemorrhage or microaneurysm.
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    tp = (pred_flat * target_flat).sum()
    fn = (target_flat * (1 - pred_flat)).sum()
    return (tp + smooth) / (tp + fn + smooth)


def specificity(pred, target, smooth=1e-6):
    """
    Specificity (True Negative Rate)
    = TN / (TN + FP)

    "Of all background pixels, how many did we correctly leave as background?"
    High specificity means fewer false alarms.
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    tn = ((1 - pred_flat) * (1 - target_flat)).sum()
    fp = (pred_flat * (1 - target_flat)).sum()
    return (tn + smooth) / (tn + fp + smooth)


def precision_score(pred, target, smooth=1e-6):
    """
    Precision (Positive Predictive Value)
    = TP / (TP + FP)

    "Of everything our model labeled as vessel, how much actually was vessel?"
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    tp = (pred_flat * target_flat).sum()
    fp = (pred_flat * (1 - target_flat)).sum()
    return (tp + smooth) / (tp + fp + smooth)


def pixel_accuracy(pred, target):
    """
    Standard pixel-wise accuracy. Included for completeness but NOT
    reliable for segmentation due to class imbalance.
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()
    correct = (pred_flat == target_flat).sum()
    total = target_flat.numel()
    return correct.float() / total


# ============================================================
# Threshold-Independent Metrics (operate on probability maps)
# ============================================================

def compute_auc_roc(pred_probs, targets):
    """
    AUC-ROC: Area Under the Receiver Operating Characteristic Curve.

    Measures overall discriminative ability across ALL possible thresholds.
    A value of 0.5 means random guessing; 1.0 means perfect separation.

    Args:
        pred_probs: Raw sigmoid probabilities (numpy, flattened)
        targets: Binary ground truth (numpy, flattened)
    """
    try:
        return roc_auc_score(targets, pred_probs)
    except ValueError:
        return 0.0


def compute_auc_pr(pred_probs, targets):
    """
    AUC-PR: Area Under the Precision-Recall Curve.

    More informative than AUC-ROC for imbalanced datasets (which vessel
    segmentation always is — vessels are ~10% of pixels).
    """
    try:
        return average_precision_score(targets, pred_probs)
    except ValueError:
        return 0.0


# ============================================================
# Batch Evaluation Helper
# ============================================================

def evaluate_batch(pred_logits, targets, threshold=0.5):
    """
    Compute ALL metrics for a single batch.

    Args:
        pred_logits: Raw model output BEFORE sigmoid (B, 1, H, W)
        targets: Binary ground truth (B, 1, H, W)
        threshold: Binarization threshold for the sigmoid output

    Returns:
        dict with all metric values
    """
    with torch.no_grad():
        pred_probs = torch.sigmoid(pred_logits)
        pred_binary = (pred_probs > threshold).float()

        metrics = {
            'dice': dice_coefficient(pred_binary, targets).item(),
            'iou': iou_score(pred_binary, targets).item(),
            'sensitivity': sensitivity(pred_binary, targets).item(),
            'specificity': specificity(pred_binary, targets).item(),
            'precision': precision_score(pred_binary, targets).item(),
            'accuracy': pixel_accuracy(pred_binary, targets).item(),
        }

    return metrics


def evaluate_full(pred_probs_all, targets_all, threshold=0.5):
    """
    Compute ALL metrics including AUC on the full dataset (after collecting
    all predictions across batches).

    Args:
        pred_probs_all: numpy array of all sigmoid probabilities (flattened)
        targets_all: numpy array of all binary targets (flattened)
    """
    pred_binary = (pred_probs_all > threshold).astype(np.float32)

    # Pixel-level metrics
    tp = (pred_binary * targets_all).sum()
    fp = (pred_binary * (1 - targets_all)).sum()
    fn = ((1 - pred_binary) * targets_all).sum()
    tn = ((1 - pred_binary) * (1 - targets_all)).sum()

    smooth = 1e-6
    dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    iou = (tp + smooth) / (tp + fp + fn + smooth)
    sens = (tp + smooth) / (tp + fn + smooth)
    spec = (tn + smooth) / (tn + fp + smooth)
    prec = (tp + smooth) / (tp + fp + smooth)
    acc = (tp + tn) / (tp + tn + fp + fn)

    # AUC metrics
    auc_roc = compute_auc_roc(pred_probs_all, targets_all)
    auc_pr = compute_auc_pr(pred_probs_all, targets_all)

    return {
        'dice': dice,
        'iou': iou,
        'sensitivity': sens,
        'specificity': spec,
        'precision': prec,
        'accuracy': acc,
        'auc_roc': auc_roc,
        'auc_pr': auc_pr,
    }



# 3 from src.trainer import DiceBCELoss, train_model
"""
Retinal DR Detection - Generic Training Engine
================================================
Provides a reusable training loop with:
- Mixed precision training (faster on modern GPUs)
- Learning rate scheduling (OneCycleLR / ReduceLROnPlateau)
- Early stopping (saves GPU hours when model plateaus)
- Model checkpointing (saves best model based on val Dice)
- Metric tracking per epoch for later plotting
"""

import os
import time
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from tqdm.notebook import tqdm




# ============================================================
# Loss Functions
# ============================================================

class DiceLoss(nn.Module):
    """
    Dice Loss = 1 - Dice Coefficient

    Directly optimizes the Dice score, which is our primary evaluation
    metric. Works much better than BCE alone for imbalanced segmentation.
    """
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred_logits, targets):
        pred = torch.sigmoid(pred_logits)
        pred_flat = pred.contiguous().view(-1)
        target_flat = targets.contiguous().view(-1)

        intersection = (pred_flat * target_flat).sum()
        dice = (2.0 * intersection + self.smooth) / (
            pred_flat.sum() + target_flat.sum() + self.smooth
        )
        return 1.0 - dice


class DiceBCELoss(nn.Module):
    """
    Combined Dice Loss + Binary Cross Entropy Loss.

    Why combine?
    - BCE provides stable gradients at the pixel level (good for learning)
    - Dice directly optimizes our evaluation metric (good for performance)
    - Together they converge faster and reach better optima.

    This is the most commonly used loss for medical image segmentation.
    """
    def __init__(self, dice_weight=0.5, bce_weight=0.5, smooth=1e-6):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.dice_loss = DiceLoss(smooth)
        self.bce_loss = nn.BCEWithLogitsLoss()

    def forward(self, pred_logits, targets):
        dice = self.dice_loss(pred_logits, targets)
        bce = self.bce_loss(pred_logits, targets)
        return self.dice_weight * dice + self.bce_weight * bce


# ============================================================
# Training & Validation Steps
# ============================================================

def train_one_epoch(model, dataloader, optimizer, criterion, device, scaler=None):
    """
    Train for one epoch.

    Args:
        model: Segmentation model
        dataloader: Training DataLoader
        optimizer: Adam/AdamW optimizer
        criterion: Loss function (DiceBCELoss)
        device: cuda/cpu
        scaler: GradScaler for mixed precision (optional)

    Returns:
        dict with average loss and metrics for this epoch
    """
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {'dice': 0, 'iou': 0, 'sensitivity': 0, 'specificity': 0}
    num_batches = 0

    for images, masks in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        # Mixed precision forward pass
        if scaler is not None:
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

        # Track metrics
        batch_metrics = evaluate_batch(outputs, masks)
        epoch_loss += loss.item()
        for k in epoch_metrics:
            epoch_metrics[k] += batch_metrics[k]
        num_batches += 1

    # Average over batches
    epoch_loss /= num_batches
    for k in epoch_metrics:
        epoch_metrics[k] /= num_batches
    epoch_metrics['loss'] = epoch_loss

    return epoch_metrics


def validate(model, dataloader, criterion, device):
    """
    Validate the model (no gradient computation).

    Returns:
        dict with average loss and metrics for validation set
    """
    model.eval()
    epoch_loss = 0.0
    epoch_metrics = {'dice': 0, 'iou': 0, 'sensitivity': 0, 'specificity': 0}
    num_batches = 0

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Validating", leave=False):
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            batch_metrics = evaluate_batch(outputs, masks)
            epoch_loss += loss.item()
            for k in epoch_metrics:
                epoch_metrics[k] += batch_metrics[k]
            num_batches += 1

    epoch_loss /= num_batches
    for k in epoch_metrics:
        epoch_metrics[k] /= num_batches
    epoch_metrics['loss'] = epoch_loss

    return epoch_metrics


# ============================================================
# Full Training Loop
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    num_epochs=50,
    scheduler=None,
    early_stopping_patience=10,
    save_dir='checkpoints',
    model_name='model',
    use_amp=True,
):
    """
    Complete training loop with early stopping and checkpointing.

    Args:
        model: Segmentation model
        train_loader: Training DataLoader
        val_loader: Validation DataLoader
        optimizer: Optimizer (e.g., AdamW)
        criterion: Loss function
        device: 'cuda' or 'cpu'
        num_epochs: Maximum number of epochs
        scheduler: LR scheduler (optional)
        early_stopping_patience: Stop if val Dice doesn't improve for N epochs
        save_dir: Directory to save model checkpoints
        model_name: Name prefix for saved checkpoint files
        use_amp: Whether to use automatic mixed precision

    Returns:
        history: dict with lists of train/val metrics per epoch
        best_model_path: path to the best saved model
    """
    os.makedirs(save_dir, exist_ok=True)

    scaler = GradScaler() if (use_amp and device.type == 'cuda') else None

    history = {
        'train_loss': [], 'val_loss': [],
        'train_dice': [], 'val_dice': [],
        'train_iou': [], 'val_iou': [],
        'train_sensitivity': [], 'val_sensitivity': [],
        'train_specificity': [], 'val_specificity': [],
    }

    best_val_dice = 0.0
    best_epoch = 0
    patience_counter = 0
    best_model_path = os.path.join(save_dir, f'{model_name}_best.pth')

    print(f"\n{'='*60}")
    print(f"Training {model_name} for up to {num_epochs} epochs")
    print(f"Device: {device} | AMP: {use_amp} | Patience: {early_stopping_patience}")
    print(f"{'='*60}\n")

    for epoch in range(1, num_epochs + 1):
        start_time = time.time()

        # Train
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)

        # Validate
        val_metrics = validate(model, val_loader, criterion, device)

        # Update scheduler
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_metrics['dice'])
            else:
                scheduler.step()

        # Record history
        for key in ['loss', 'dice', 'iou', 'sensitivity', 'specificity']:
            history[f'train_{key}'].append(train_metrics[key])
            history[f'val_{key}'].append(val_metrics[key])

        elapsed = time.time() - start_time
        current_lr = optimizer.param_groups[0]['lr']

        # Print epoch summary
        print(
            f"Epoch {epoch:3d}/{num_epochs} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val Dice: {val_metrics['dice']:.4f} | "
            f"Val IoU: {val_metrics['iou']:.4f} | "
            f"LR: {current_lr:.2e} | "
            f"Time: {elapsed:.1f}s"
        )

        # Check for improvement
        if val_metrics['dice'] > best_val_dice:
            best_val_dice = val_metrics['dice']
            best_epoch = epoch
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_dice': best_val_dice,
                'history': history,
            }, best_model_path)
            print(f"  ★ New best model saved! Dice: {best_val_dice:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"\n⏹ Early stopping at epoch {epoch}. "
                      f"Best Dice: {best_val_dice:.4f} at epoch {best_epoch}")
                break

    print(f"\n{'='*60}")
    print(f"Training complete. Best Val Dice: {best_val_dice:.4f} (epoch {best_epoch})")
    print(f"Best model saved to: {best_model_path}")
    print(f"{'='*60}")

    return history, best_model_path


# 4 from src.visualize import plot_training_history, plot_predictions, plot_roc_pr_curves

"""
Retinal DR Detection - Visualization Utilities
================================================
Provides functions for:
- Training history curves (loss, Dice, IoU over epochs)
- Prediction overlays (original image + ground truth + model prediction)
- Side-by-side model comparison charts
- Confusion matrix display
"""

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_curve, precision_recall_curve


# ============================================================
# Training History Plots
# ============================================================

def plot_training_history(history, model_name='Model'):
    """
    Plot loss and all key metrics over training epochs.

    Creates a 2x2 grid:
    - Top-left: Loss (train vs val)
    - Top-right: Dice (train vs val)
    - Bottom-left: IoU (train vs val)
    - Bottom-right: Sensitivity & Specificity (val only)
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{model_name} — Training History', fontsize=16, fontweight='bold')
    epochs = range(1, len(history['train_loss']) + 1)

    # Loss
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train')
    axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Val')
    axes[0, 0].set_title('Loss (DiceBCE)')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Dice
    axes[0, 1].plot(epochs, history['train_dice'], 'b-', label='Train')
    axes[0, 1].plot(epochs, history['val_dice'], 'r-', label='Val')
    axes[0, 1].set_title('Dice Coefficient')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # IoU
    axes[1, 0].plot(epochs, history['train_iou'], 'b-', label='Train')
    axes[1, 0].plot(epochs, history['val_iou'], 'r-', label='Val')
    axes[1, 0].set_title('IoU (Jaccard)')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Sensitivity & Specificity
    axes[1, 1].plot(epochs, history['val_sensitivity'], 'g-', label='Val Sensitivity')
    axes[1, 1].plot(epochs, history['val_specificity'], 'm-', label='Val Specificity')
    axes[1, 1].set_title('Sensitivity & Specificity (Val)')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig


# ============================================================
# Prediction Visualization
# ============================================================

def denormalize(tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    tensor = tensor.cpu() * std + mean
    return torch.clamp(tensor, 0, 1).permute(1, 2, 0).numpy()


def plot_predictions(model, dataloader, device, num_samples=4, threshold=0.5, model_name='Model'):
    """
    Visualize model predictions on validation/test data.

    For each sample shows 4 columns:
    1. Original fundus image
    2. Ground truth vessel mask
    3. Model's predicted mask
    4. Overlay (original + prediction in red + ground truth in green)
    """
    model.eval()
    images_shown = 0

    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))
    fig.suptitle(f'{model_name} — Predictions', fontsize=16, fontweight='bold')

    if num_samples == 1:
        axes = axes.reshape(1, -1)

    col_titles = ['Original Image', 'Ground Truth', 'Prediction', 'Overlay']
    for j, title in enumerate(col_titles):
        axes[0, j].set_title(title, fontsize=12, fontweight='bold')

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.sigmoid(outputs)
            preds_binary = (preds > threshold).float()

            for i in range(images.shape[0]):
                if images_shown >= num_samples:
                    break

                img_np = denormalize(images[i])
                gt_np = masks[i].squeeze().cpu().numpy()
                pred_np = preds_binary[i].squeeze().cpu().numpy()

                # Original image
                axes[images_shown, 0].imshow(img_np)
                axes[images_shown, 0].axis('off')

                # Ground truth
                axes[images_shown, 1].imshow(gt_np, cmap='gray')
                axes[images_shown, 1].axis('off')

                # Prediction
                axes[images_shown, 2].imshow(pred_np, cmap='gray')
                axes[images_shown, 2].axis('off')

                # Overlay: Green = GT, Red = Prediction
                overlay = img_np.copy()
                overlay[gt_np > 0.5] = [0, 1, 0]      # Green for ground truth
                overlay[pred_np > 0.5] = [1, 0, 0]     # Red for prediction
                # Yellow where both overlap (correct predictions)
                both = (gt_np > 0.5) & (pred_np > 0.5)
                overlay[both] = [1, 1, 0]
                axes[images_shown, 3].imshow(overlay)
                axes[images_shown, 3].axis('off')

                images_shown += 1

            if images_shown >= num_samples:
                break

    plt.tight_layout()
    plt.show()
    return fig


# ============================================================
# ROC and PR Curve Plots
# ============================================================

def plot_roc_pr_curves(pred_probs, targets, model_name='Model'):
    """
    Plot ROC curve and Precision-Recall curve side by side.

    Args:
        pred_probs: Flattened numpy array of sigmoid probabilities
        targets: Flattened numpy array of binary ground truth
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name} — ROC & PR Curves', fontsize=14, fontweight='bold')

    # ROC Curve
    fpr, tpr, _ = roc_curve(targets, pred_probs)
    ax1.plot(fpr, tpr, 'b-', linewidth=2)
    ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax1.set_xlabel('False Positive Rate')
    ax1.set_ylabel('True Positive Rate')
    ax1.set_title('ROC Curve')
    ax1.grid(True, alpha=0.3)

    # PR Curve
    precision, recall, _ = precision_recall_curve(targets, pred_probs)
    ax2.plot(recall, precision, 'r-', linewidth=2)
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title('Precision-Recall Curve')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig


# ============================================================
# Model Comparison Table
# ============================================================

def print_comparison_table(results_dict):
    """
    Print a formatted comparison table of all models.

    Args:
        results_dict: dict of {model_name: {metric_name: value}}

    Example:
        results = {
            'U-Net': {'dice': 0.82, 'iou': 0.70, ...},
            'TransUNet': {'dice': 0.85, 'iou': 0.74, ...},
        }
    """
    metrics = ['dice', 'iou', 'sensitivity', 'specificity', 'precision', 'accuracy', 'auc_roc', 'auc_pr']
    header = f"{'Model':<20}" + "".join([f"{m:>14}" for m in metrics])
    print("=" * len(header))
    print(header)
    print("=" * len(header))

    for model_name, scores in results_dict.items():
        row = f"{model_name:<20}"
        for m in metrics:
            val = scores.get(m, 0)
            row += f"{val:>14.4f}"
        print(row)

    print("=" * len(header))

    # Highlight best model per metric
    print("\n★ Best per metric:")
    for m in metrics:
        best_model = max(results_dict, key=lambda x: results_dict[x].get(m, 0))
        best_val = results_dict[best_model].get(m, 0)
        print(f"  {m:>14}: {best_model} ({best_val:.4f})")
# Our shared modules sice we are going to provide them in github as moduls and python functions we will write the here

# 1 from src.dataset import RetinalDataset, get_train_transforms, get_val_transforms

"""
Retinal DR Detection - PyTorch Dataset & Augmentation Pipelines
================================================================
This module provides the RetinalDataset class that loads preprocessed
fundus images and their corresponding vessel/lesion masks.

It uses Albumentations for on-the-fly augmentation, ensuring that
geometric transforms (flip, rotate, distort) are applied identically
to both the image and its mask — critical for segmentation tasks.
"""

import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2


class RetinalDataset(Dataset):
    """
    PyTorch Dataset for retinal fundus images and segmentation masks.

    This dataset reads image-mask pairs from a CSV file generated by
    the preprocessing notebook (01_preprocessing.ipynb). Each row in
    the CSV contains relative paths to the processed image and its
    corresponding vessel mask.

    Args:
        csv_file (str): Path to the CSV split file (train/val/test).
        base_dir (str): Root directory of the processed dataset.
        transform (callable): Albumentations transform pipeline.
        mask_col (str): Column name in CSV pointing to the mask path.
                        Default 'vessel_path' for vessel segmentation.
    """

    def __init__(self, csv_file, base_dir, transform=None, mask_col='vessel_path'):
        self.df = pd.read_csv(csv_file)
        self.base_dir = base_dir
        self.transform = transform
        self.mask_col = mask_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Build full paths
        img_path = os.path.join(self.base_dir, row['img_path'])
        mask_path = os.path.join(self.base_dir, row[self.mask_col])

        # Read image (BGR -> RGB for Albumentations)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Read mask (grayscale, binary)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)  # Binarize to 0.0 / 1.0

        # Apply augmentations (same transform applied to BOTH image and mask)
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # Ensure mask has channel dimension: (1, H, W)
        if isinstance(mask, torch.Tensor):
            mask = mask.unsqueeze(0)
        else:
            mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask


# ============================================================
# Augmentation Pipelines
# ============================================================

def get_train_transforms(img_size=512):
    """
    Training augmentation pipeline.

    Geometric transforms are critical for retinal images because:
    - The retina can be photographed from any angle -> flips & rotations
    - Different cameras have different FOVs -> scale variations
    - Slight patient movement -> shift/distortion

    Color transforms handle:
    - Different fundus camera brands (Topcon vs Zeiss vs Canon)
    - Different illumination conditions
    """
    return A.Compose([
        A.Resize(img_size, img_size),

        # === Geometric Augmentations ===
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.0625,
            scale_limit=0.1,
            rotate_limit=45,
            border_mode=cv2.BORDER_CONSTANT,
            p=0.5
        ),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
        A.ElasticTransform(alpha=120, sigma=120 * 0.05, p=0.2),

        # === Color / Illumination Augmentations ===
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(var_limit=(5, 25), p=0.2),

        # === Normalization (ImageNet stats for pretrained backbones) ===
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


def get_val_transforms(img_size=512):
    """
    Validation/Test transform — NO augmentation, only resize + normalize.
    We must evaluate on clean, unaugmented images for fair comparison.
    """
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

# 2 from src.metrics import evaluate_full
"""
Retinal DR Detection - Medical Evaluation Metrics
===================================================
Comprehensive metrics for evaluating segmentation quality in clinical context.

In medical imaging, standard accuracy is NOT enough. A model that predicts
"no vessel everywhere" would get ~90% accuracy (because vessels are only ~10%
of the image), but would be completely useless clinically.

Key metrics explained:
- Dice Coefficient: Overlap between prediction and ground truth (harmonic mean of P & R)
- IoU (Jaccard): Intersection over Union — stricter than Dice
- Sensitivity (Recall): "Of all actual vessels, how many did we detect?"
  → Critical in medicine: missing a vessel = missing pathology
- Specificity: "Of all non-vessel pixels, how many did we correctly identify?"
  → Important to avoid false alarms
- Precision (PPV): "Of everything we predicted as vessel, how much was correct?"
- AUC-ROC: Overall discriminative ability across all thresholds
- AUC-PR: Better than AUC-ROC for imbalanced data (which vessel segmentation is)
"""

import numpy as np
import torch
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix


# ============================================================
# Pixel-Level Metrics (operate on binary masks)
# ============================================================

def dice_coefficient(pred, target, smooth=1e-6):
    """
    Dice = 2 * |P ∩ T| / (|P| + |T|)

    Ranges from 0 (no overlap) to 1 (perfect overlap).
    This is THE standard metric for medical image segmentation.

    Args:
        pred: Binary prediction tensor (B, 1, H, W)
        target: Binary ground truth tensor (B, 1, H, W)
        smooth: Smoothing factor to avoid division by zero
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    intersection = (pred_flat * target_flat).sum()
    return (2.0 * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)


def iou_score(pred, target, smooth=1e-6):
    """
    IoU (Jaccard Index) = |P ∩ T| / |P ∪ T|

    Stricter than Dice — penalizes false positives and false negatives more.
    IoU of 0.5 means the prediction overlaps only half the ground truth area.
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    intersection = (pred_flat * target_flat).sum()
    union = pred_flat.sum() + target_flat.sum() - intersection
    return (intersection + smooth) / (union + smooth)


def sensitivity(pred, target, smooth=1e-6):
    """
    Sensitivity (Recall / True Positive Rate)
    = TP / (TP + FN)

    "Of all real vessels in the image, what fraction did our model find?"
    In clinical settings this is CRITICAL — a missed vessel could mean
    missing a hemorrhage or microaneurysm.
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    tp = (pred_flat * target_flat).sum()
    fn = (target_flat * (1 - pred_flat)).sum()
    return (tp + smooth) / (tp + fn + smooth)


def specificity(pred, target, smooth=1e-6):
    """
    Specificity (True Negative Rate)
    = TN / (TN + FP)

    "Of all background pixels, how many did we correctly leave as background?"
    High specificity means fewer false alarms.
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    tn = ((1 - pred_flat) * (1 - target_flat)).sum()
    fp = (pred_flat * (1 - target_flat)).sum()
    return (tn + smooth) / (tn + fp + smooth)


def precision_score(pred, target, smooth=1e-6):
    """
    Precision (Positive Predictive Value)
    = TP / (TP + FP)

    "Of everything our model labeled as vessel, how much actually was vessel?"
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()

    tp = (pred_flat * target_flat).sum()
    fp = (pred_flat * (1 - target_flat)).sum()
    return (tp + smooth) / (tp + fp + smooth)


def pixel_accuracy(pred, target):
    """
    Standard pixel-wise accuracy. Included for completeness but NOT
    reliable for segmentation due to class imbalance.
    """
    pred_flat = pred.contiguous().view(-1).float()
    target_flat = target.contiguous().view(-1).float()
    correct = (pred_flat == target_flat).sum()
    total = target_flat.numel()
    return correct.float() / total


# ============================================================
# Threshold-Independent Metrics (operate on probability maps)
# ============================================================

def compute_auc_roc(pred_probs, targets):
    """
    AUC-ROC: Area Under the Receiver Operating Characteristic Curve.

    Measures overall discriminative ability across ALL possible thresholds.
    A value of 0.5 means random guessing; 1.0 means perfect separation.

    Args:
        pred_probs: Raw sigmoid probabilities (numpy, flattened)
        targets: Binary ground truth (numpy, flattened)
    """
    try:
        return roc_auc_score(targets, pred_probs)
    except ValueError:
        return 0.0


def compute_auc_pr(pred_probs, targets):
    """
    AUC-PR: Area Under the Precision-Recall Curve.

    More informative than AUC-ROC for imbalanced datasets (which vessel
    segmentation always is — vessels are ~10% of pixels).
    """
    try:
        return average_precision_score(targets, pred_probs)
    except ValueError:
        return 0.0


# ============================================================
# Batch Evaluation Helper
# ============================================================

def evaluate_batch(pred_logits, targets, threshold=0.5):
    """
    Compute ALL metrics for a single batch.

    Args:
        pred_logits: Raw model output BEFORE sigmoid (B, 1, H, W)
        targets: Binary ground truth (B, 1, H, W)
        threshold: Binarization threshold for the sigmoid output

    Returns:
        dict with all metric values
    """
    with torch.no_grad():
        pred_probs = torch.sigmoid(pred_logits)
        pred_binary = (pred_probs > threshold).float()

        metrics = {
            'dice': dice_coefficient(pred_binary, targets).item(),
            'iou': iou_score(pred_binary, targets).item(),
            'sensitivity': sensitivity(pred_binary, targets).item(),
            'specificity': specificity(pred_binary, targets).item(),
            'precision': precision_score(pred_binary, targets).item(),
            'accuracy': pixel_accuracy(pred_binary, targets).item(),
        }

    return metrics


def evaluate_full(pred_probs_all, targets_all, threshold=0.5):
    """
    Compute ALL metrics including AUC on the full dataset (after collecting
    all predictions across batches).

    Args:
        pred_probs_all: numpy array of all sigmoid probabilities (flattened)
        targets_all: numpy array of all binary targets (flattened)
    """
    pred_binary = (pred_probs_all > threshold).astype(np.float32)

    # Pixel-level metrics
    tp = (pred_binary * targets_all).sum()
    fp = (pred_binary * (1 - targets_all)).sum()
    fn = ((1 - pred_binary) * targets_all).sum()
    tn = ((1 - pred_binary) * (1 - targets_all)).sum()

    smooth = 1e-6
    dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    iou = (tp + smooth) / (tp + fp + fn + smooth)
    sens = (tp + smooth) / (tp + fn + smooth)
    spec = (tn + smooth) / (tn + fp + smooth)
    prec = (tp + smooth) / (tp + fp + smooth)
    acc = (tp + tn) / (tp + tn + fp + fn)

    # AUC metrics
    auc_roc = compute_auc_roc(pred_probs_all, targets_all)
    auc_pr = compute_auc_pr(pred_probs_all, targets_all)

    return {
        'dice': dice,
        'iou': iou,
        'sensitivity': sens,
        'specificity': spec,
        'precision': prec,
        'accuracy': acc,
        'auc_roc': auc_roc,
        'auc_pr': auc_pr,
    }



# 3 from src.trainer import DiceBCELoss, train_model
"""
Retinal DR Detection - Generic Training Engine
================================================
Provides a reusable training loop with:
- Mixed precision training (faster on modern GPUs)
- Learning rate scheduling (OneCycleLR / ReduceLROnPlateau)
- Early stopping (saves GPU hours when model plateaus)
- Model checkpointing (saves best model based on val Dice)
- Metric tracking per epoch for later plotting
"""

import os
import time
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from tqdm.notebook import tqdm




# ============================================================
# Loss Functions
# ============================================================

class DiceLoss(nn.Module):
    """
    Dice Loss = 1 - Dice Coefficient

    Directly optimizes the Dice score, which is our primary evaluation
    metric. Works much better than BCE alone for imbalanced segmentation.
    """
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred_logits, targets):
        pred = torch.sigmoid(pred_logits)
        pred_flat = pred.contiguous().view(-1)
        target_flat = targets.contiguous().view(-1)

        intersection = (pred_flat * target_flat).sum()
        dice = (2.0 * intersection + self.smooth) / (
            pred_flat.sum() + target_flat.sum() + self.smooth
        )
        return 1.0 - dice


class DiceBCELoss(nn.Module):
    """
    Combined Dice Loss + Binary Cross Entropy Loss.

    Why combine?
    - BCE provides stable gradients at the pixel level (good for learning)
    - Dice directly optimizes our evaluation metric (good for performance)
    - Together they converge faster and reach better optima.

    This is the most commonly used loss for medical image segmentation.
    """
    def __init__(self, dice_weight=0.5, bce_weight=0.5, smooth=1e-6):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.dice_loss = DiceLoss(smooth)
        self.bce_loss = nn.BCEWithLogitsLoss()

    def forward(self, pred_logits, targets):
        dice = self.dice_loss(pred_logits, targets)
        bce = self.bce_loss(pred_logits, targets)
        return self.dice_weight * dice + self.bce_weight * bce


# ============================================================
# Training & Validation Steps
# ============================================================

def train_one_epoch(model, dataloader, optimizer, criterion, device, scaler=None):
    """
    Train for one epoch.

    Args:
        model: Segmentation model
        dataloader: Training DataLoader
        optimizer: Adam/AdamW optimizer
        criterion: Loss function (DiceBCELoss)
        device: cuda/cpu
        scaler: GradScaler for mixed precision (optional)

    Returns:
        dict with average loss and metrics for this epoch
    """
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {'dice': 0, 'iou': 0, 'sensitivity': 0, 'specificity': 0}
    num_batches = 0

    for images, masks in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        # Mixed precision forward pass
        if scaler is not None:
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

        # Track metrics
        batch_metrics = evaluate_batch(outputs, masks)
        epoch_loss += loss.item()
        for k in epoch_metrics:
            epoch_metrics[k] += batch_metrics[k]
        num_batches += 1

    # Average over batches
    epoch_loss /= num_batches
    for k in epoch_metrics:
        epoch_metrics[k] /= num_batches
    epoch_metrics['loss'] = epoch_loss

    return epoch_metrics


def validate(model, dataloader, criterion, device):
    """
    Validate the model (no gradient computation).

    Returns:
        dict with average loss and metrics for validation set
    """
    model.eval()
    epoch_loss = 0.0
    epoch_metrics = {'dice': 0, 'iou': 0, 'sensitivity': 0, 'specificity': 0}
    num_batches = 0

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Validating", leave=False):
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            batch_metrics = evaluate_batch(outputs, masks)
            epoch_loss += loss.item()
            for k in epoch_metrics:
                epoch_metrics[k] += batch_metrics[k]
            num_batches += 1

    epoch_loss /= num_batches
    for k in epoch_metrics:
        epoch_metrics[k] /= num_batches
    epoch_metrics['loss'] = epoch_loss

    return epoch_metrics


# ============================================================
# Full Training Loop
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    num_epochs=50,
    scheduler=None,
    early_stopping_patience=10,
    save_dir='checkpoints',
    model_name='model',
    use_amp=True,
):
    """
    Complete training loop with early stopping and checkpointing.

    Args:
        model: Segmentation model
        train_loader: Training DataLoader
        val_loader: Validation DataLoader
        optimizer: Optimizer (e.g., AdamW)
        criterion: Loss function
        device: 'cuda' or 'cpu'
        num_epochs: Maximum number of epochs
        scheduler: LR scheduler (optional)
        early_stopping_patience: Stop if val Dice doesn't improve for N epochs
        save_dir: Directory to save model checkpoints
        model_name: Name prefix for saved checkpoint files
        use_amp: Whether to use automatic mixed precision

    Returns:
        history: dict with lists of train/val metrics per epoch
        best_model_path: path to the best saved model
    """
    os.makedirs(save_dir, exist_ok=True)

    scaler = GradScaler() if (use_amp and device.type == 'cuda') else None

    history = {
        'train_loss': [], 'val_loss': [],
        'train_dice': [], 'val_dice': [],
        'train_iou': [], 'val_iou': [],
        'train_sensitivity': [], 'val_sensitivity': [],
        'train_specificity': [], 'val_specificity': [],
    }

    best_val_dice = 0.0
    best_epoch = 0
    patience_counter = 0
    best_model_path = os.path.join(save_dir, f'{model_name}_best.pth')

    print(f"\n{'='*60}")
    print(f"Training {model_name} for up to {num_epochs} epochs")
    print(f"Device: {device} | AMP: {use_amp} | Patience: {early_stopping_patience}")
    print(f"{'='*60}\n")

    for epoch in range(1, num_epochs + 1):
        start_time = time.time()

        # Train
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)

        # Validate
        val_metrics = validate(model, val_loader, criterion, device)

        # Update scheduler
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_metrics['dice'])
            else:
                scheduler.step()

        # Record history
        for key in ['loss', 'dice', 'iou', 'sensitivity', 'specificity']:
            history[f'train_{key}'].append(train_metrics[key])
            history[f'val_{key}'].append(val_metrics[key])

        elapsed = time.time() - start_time
        current_lr = optimizer.param_groups[0]['lr']

        # Print epoch summary
        print(
            f"Epoch {epoch:3d}/{num_epochs} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val Dice: {val_metrics['dice']:.4f} | "
            f"Val IoU: {val_metrics['iou']:.4f} | "
            f"LR: {current_lr:.2e} | "
            f"Time: {elapsed:.1f}s"
        )

        # Check for improvement
        if val_metrics['dice'] > best_val_dice:
            best_val_dice = val_metrics['dice']
            best_epoch = epoch
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_dice': best_val_dice,
                'history': history,
            }, best_model_path)
            print(f"  ★ New best model saved! Dice: {best_val_dice:.4f}")
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"\n⏹ Early stopping at epoch {epoch}. "
                      f"Best Dice: {best_val_dice:.4f} at epoch {best_epoch}")
                break

    print(f"\n{'='*60}")
    print(f"Training complete. Best Val Dice: {best_val_dice:.4f} (epoch {best_epoch})")
    print(f"Best model saved to: {best_model_path}")
    print(f"{'='*60}")

    return history, best_model_path


# 4 from src.visualize import plot_training_history, plot_predictions, plot_roc_pr_curves

"""
Retinal DR Detection - Visualization Utilities
================================================
Provides functions for:
- Training history curves (loss, Dice, IoU over epochs)
- Prediction overlays (original image + ground truth + model prediction)
- Side-by-side model comparison charts
- Confusion matrix display
"""

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_curve, precision_recall_curve


# ============================================================
# Training History Plots
# ============================================================

def plot_training_history(history, model_name='Model'):
    """
    Plot loss and all key metrics over training epochs.

    Creates a 2x2 grid:
    - Top-left: Loss (train vs val)
    - Top-right: Dice (train vs val)
    - Bottom-left: IoU (train vs val)
    - Bottom-right: Sensitivity & Specificity (val only)
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{model_name} — Training History', fontsize=16, fontweight='bold')
    epochs = range(1, len(history['train_loss']) + 1)

    # Loss
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train')
    axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Val')
    axes[0, 0].set_title('Loss (DiceBCE)')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Dice
    axes[0, 1].plot(epochs, history['train_dice'], 'b-', label='Train')
    axes[0, 1].plot(epochs, history['val_dice'], 'r-', label='Val')
    axes[0, 1].set_title('Dice Coefficient')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # IoU
    axes[1, 0].plot(epochs, history['train_iou'], 'b-', label='Train')
    axes[1, 0].plot(epochs, history['val_iou'], 'r-', label='Val')
    axes[1, 0].set_title('IoU (Jaccard)')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Sensitivity & Specificity
    axes[1, 1].plot(epochs, history['val_sensitivity'], 'g-', label='Val Sensitivity')
    axes[1, 1].plot(epochs, history['val_specificity'], 'm-', label='Val Specificity')
    axes[1, 1].set_title('Sensitivity & Specificity (Val)')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig


# ============================================================
# Prediction Visualization
# ============================================================

def denormalize(tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    tensor = tensor.cpu() * std + mean
    return torch.clamp(tensor, 0, 1).permute(1, 2, 0).numpy()


def plot_predictions(model, dataloader, device, num_samples=4, threshold=0.5, model_name='Model'):
    """
    Visualize model predictions on validation/test data.

    For each sample shows 4 columns:
    1. Original fundus image
    2. Ground truth vessel mask
    3. Model's predicted mask
    4. Overlay (original + prediction in red + ground truth in green)
    """
    model.eval()
    images_shown = 0

    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))
    fig.suptitle(f'{model_name} — Predictions', fontsize=16, fontweight='bold')

    if num_samples == 1:
        axes = axes.reshape(1, -1)

    col_titles = ['Original Image', 'Ground Truth', 'Prediction', 'Overlay']
    for j, title in enumerate(col_titles):
        axes[0, j].set_title(title, fontsize=12, fontweight='bold')

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.sigmoid(outputs)
            preds_binary = (preds > threshold).float()

            for i in range(images.shape[0]):
                if images_shown >= num_samples:
                    break

                img_np = denormalize(images[i])
                gt_np = masks[i].squeeze().cpu().numpy()
                pred_np = preds_binary[i].squeeze().cpu().numpy()

                # Original image
                axes[images_shown, 0].imshow(img_np)
                axes[images_shown, 0].axis('off')

                # Ground truth
                axes[images_shown, 1].imshow(gt_np, cmap='gray')
                axes[images_shown, 1].axis('off')

                # Prediction
                axes[images_shown, 2].imshow(pred_np, cmap='gray')
                axes[images_shown, 2].axis('off')

                # Overlay: Green = GT, Red = Prediction
                overlay = img_np.copy()
                overlay[gt_np > 0.5] = [0, 1, 0]      # Green for ground truth
                overlay[pred_np > 0.5] = [1, 0, 0]     # Red for prediction
                # Yellow where both overlap (correct predictions)
                both = (gt_np > 0.5) & (pred_np > 0.5)
                overlay[both] = [1, 1, 0]
                axes[images_shown, 3].imshow(overlay)
                axes[images_shown, 3].axis('off')

                images_shown += 1

            if images_shown >= num_samples:
                break

    plt.tight_layout()
    plt.show()
    return fig


# ============================================================
# ROC and PR Curve Plots
# ============================================================

def plot_roc_pr_curves(pred_probs, targets, model_name='Model'):
    """
    Plot ROC curve and Precision-Recall curve side by side.

    Args:
        pred_probs: Flattened numpy array of sigmoid probabilities
        targets: Flattened numpy array of binary ground truth
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name} — ROC & PR Curves', fontsize=14, fontweight='bold')

    # ROC Curve
    fpr, tpr, _ = roc_curve(targets, pred_probs)
    ax1.plot(fpr, tpr, 'b-', linewidth=2)
    ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax1.set_xlabel('False Positive Rate')
    ax1.set_ylabel('True Positive Rate')
    ax1.set_title('ROC Curve')
    ax1.grid(True, alpha=0.3)

    # PR Curve
    precision, recall, _ = precision_recall_curve(targets, pred_probs)
    ax2.plot(recall, precision, 'r-', linewidth=2)
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title('Precision-Recall Curve')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return fig


# ============================================================
# Model Comparison Table
# ============================================================

def print_comparison_table(results_dict):
    """
    Print a formatted comparison table of all models.

    Args:
        results_dict: dict of {model_name: {metric_name: value}}

    Example:
        results = {
            'U-Net': {'dice': 0.82, 'iou': 0.70, ...},
            'TransUNet': {'dice': 0.85, 'iou': 0.74, ...},
        }
    """
    metrics = ['dice', 'iou', 'sensitivity', 'specificity', 'precision', 'accuracy', 'auc_roc', 'auc_pr']
    header = f"{'Model':<20}" + "".join([f"{m:>14}" for m in metrics])
    print("=" * len(header))
    print(header)
    print("=" * len(header))

    for model_name, scores in results_dict.items():
        row = f"{model_name:<20}"
        for m in metrics:
            val = scores.get(m, 0)
            row += f"{val:>14.4f}"
        print(row)

    print("=" * len(header))

    # Highlight best model per metric
    print("\n★ Best per metric:")
    for m in metrics:
        best_model = max(results_dict, key=lambda x: results_dict[x].get(m, 0))
        best_val = results_dict[best_model].get(m, 0)
        print(f"  {m:>14}: {best_model} ({best_val:.4f})")


In [ ]:
import zipfile


zip_path = "DR_dataset_processed.zip"     # Change this to your actual zip file name
extract_dir = "dataset"           # Where to extract your files

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

In [ ]:
# ============================================================
# Cell 3: Configuration
# ============================================================
DATA_DIR = 'dataset'
TEST_CSV = os.path.join(DATA_DIR, 'test_split.csv')
IMG_SIZE = 512
BATCH_SIZE = 4

# Paths to saved checkpoints
# Upload these .pt files to Colab before running!
CSNET_CHECKPOINT = 'csnet_best.pt'
ATTN_UNET_CHECKPOINT = 'best_model.pt'
SWIN_UNET_CHECKPOINT = 'swin_unet_best.pt'

SAVE_DIR = 'results/ensemble'
os.makedirs(SAVE_DIR, exist_ok=True)

## Step 1: Define All Three Architectures
We need to re-define the exact same architecture classes used during training so we can load the saved weights.

**Important**: The architecture definition must EXACTLY match what was used during training, otherwise `load_state_dict()` will fail because the layer names won't match.

In [ ]:
# ============================================================
# Cell 4: CS-Net Architecture Definition
# ============================================================
# This is the EXACT same code from the CSNet training notebook.
# We must define it here so PyTorch knows the layer structure
# when loading the saved weights.

class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation style channel attention."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class SpatialAttention(nn.Module):
    """Spatial attention via channel-wise pooling."""
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg_out, max_out], dim=1)
        attn = self.conv(combined)
        return x * attn


class CSBlock(nn.Module):
    """Channel-Spatial Attention Block."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = self.ca(x)
        x = self.sa(x)
        return x


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)


class CSNet(nn.Module):
    """CS-Net: U-Net with Channel-Spatial Attention."""
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.cs_blocks = nn.ModuleList()
        self.pools = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.encoders.append(ConvBlock(ch, f))
            self.cs_blocks.append(CSBlock(f))
            self.pools.append(nn.MaxPool2d(2))
            ch = f

        self.bottleneck = ConvBlock(features[-1], features[-1] * 2)
        self.cs_bottleneck = CSBlock(features[-1] * 2)

        self.upconvs = nn.ModuleList()
        self.decoders = nn.ModuleList()
        rev = list(reversed(features))
        prev = features[-1] * 2
        for f in rev:
            self.upconvs.append(nn.ConvTranspose2d(prev, f, 2, stride=2))
            self.decoders.append(ConvBlock(f * 2, f))
            prev = f

        self.final = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for enc, cs, pool in zip(self.encoders, self.cs_blocks, self.pools):
            x = enc(x)
            x = cs(x)
            skips.append(x)
            x = pool(x)
        x = self.bottleneck(x)
        x = self.cs_bottleneck(x)
        skips = skips[::-1]
        for up, dec, skip in zip(self.upconvs, self.decoders, skips):
            x = up(x)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
        return self.final(x)

print('✅ CS-Net architecture defined')

In [ ]:
# ============================================================
# Cell 5: Swin-UNet Architecture Definition
# ============================================================

class DecoderBlockSwin(nn.Module):
    """Decoder block: upsample + concat skip + convolutions."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class SwinUNet(nn.Module):
    """Swin Transformer Encoder + CNN Decoder."""
    def __init__(self, out_channels=1, pretrained=False):
        super().__init__()
        self.encoder = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=pretrained,
            features_only=True,
            img_size=512,
        )
        self.encoder_channels = self.encoder.feature_info.channels()

        self.dec4 = DecoderBlockSwin(self.encoder_channels[3], self.encoder_channels[2], 256)
        self.dec3 = DecoderBlockSwin(256, self.encoder_channels[1], 128)
        self.dec2 = DecoderBlockSwin(128, self.encoder_channels[0], 64)

        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, stride=4),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.final_conv = nn.Conv2d(32, out_channels, 1)

    def _fix_feature_format(self, features):
        fixed = []
        for feat, expected_c in zip(features, self.encoder_channels):
            if feat.shape[1] != expected_c:
                feat = feat.permute(0, 3, 1, 2).contiguous()
            fixed.append(feat)
        return fixed

    def forward(self, x):
        features = self.encoder(x)
        features = self._fix_feature_format(features)
        d4 = self.dec4(features[3], features[2])
        d3 = self.dec3(d4, features[1])
        d2 = self.dec2(d3, features[0])
        out = self.final_up(d2)
        out = F.interpolate(out, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)
        return self.final_conv(out)

print('✅ Swin-UNet architecture defined')

## Step 2: Load the Trained Models

Each model was trained separately in its own notebook. Now we:
1. **Instantiate** the architecture (creates the layers with random weights)
2. **Load** the saved checkpoint (overwrites random weights with trained weights)
3. Set to **eval mode** (disables dropout and uses running stats for batch norm)

⚠️ **Upload your checkpoint files** to Colab before running this cell!

In [ ]:
# ============================================================
# Cell 6: Load All Three Trained Models
# ============================================================

def load_model(model, checkpoint_path, model_name):
    """
    Load a model checkpoint. Handles both formats:
    - Full checkpoint: {'model_state_dict': ..., 'optimizer_state_dict': ...}
    - Direct state_dict: {layer_name: tensor, ...}
    """

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)


    # Handle different checkpoint formats
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f'  ✅ {model_name}: Loaded from full checkpoint')
    elif 'state_dict' in checkpoint:
        model.load_state_dict(checkpoint['state_dict'])
        print(f'  ✅ {model_name}: Loaded from state_dict')
    else:
        # Assume the checkpoint IS the state_dict
        model.load_state_dict(checkpoint)
        print(f'  ✅ {model_name}: Loaded direct state_dict')

    model.to(device)
    model.eval()
    return model


# --- Model 1: CS-Net ---
csnet = CSNet(in_channels=3, out_channels=1)
csnet = load_model(csnet, CSNET_CHECKPOINT, 'CS-Net')

# --- Model 2: Attention U-Net (via segmentation-models-pytorch) ---
attn_unet = smp.Unet(
    encoder_name='resnet34',
    encoder_weights=None,         # We load our own trained weights
    decoder_attention_type='scse',
    in_channels=3,
    classes=1,
    activation=None,
)
attn_unet = load_model(attn_unet, ATTN_UNET_CHECKPOINT, 'Attention U-Net')

# --- Model 3: Swin-UNet ---
swin_unet = SwinUNet(out_channels=1, pretrained=False)  # pretrained=False since we load our weights
swin_unet = load_model(swin_unet, SWIN_UNET_CHECKPOINT, 'Swin-UNet')

print(f'\n🔥 All 3 models loaded and ready for inference!')

In [ ]:
# ============================================================
# Cell 7: Load Test Dataset
# ============================================================
test_dataset = RetinalDataset(TEST_CSV, DATA_DIR, get_val_transforms(IMG_SIZE))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)

print(f'Test set: {len(test_dataset)} images')
print(f'Batches: {len(test_loader)}')

## Step 3: Generate Predictions from All Models

We pass the entire test set through each model and collect their **raw sigmoid probability maps**.

Why raw probabilities instead of binary masks? Because averaging probabilities is much smoother — if CSNet says 0.6 (maybe vessel) and AttentionUNet says 0.8 (likely vessel), the average 0.7 still crosses the 0.5 threshold. If we binarized first, we'd lose this nuance.

In [ ]:
# ============================================================
# Cell 8: Collect Predictions from All Models
# ============================================================

def get_all_predictions(models, dataloader, device):
    """
    Run all models on the test set and collect their
    raw sigmoid probability maps.

    Returns:
        model_preds: dict of {model_name: flat numpy array of probabilities}
        all_targets: flat numpy array of ground truth labels
        batch_data: list of (images, masks, per-model probs) for visualization
    """
    model_preds = {name: [] for name in models.keys()}
    all_targets = []
    batch_data = []     # Store some batches for visualization later

    with torch.no_grad():
        for batch_idx, (images, masks) in enumerate(dataloader):
            images = images.to(device)
            batch_probs = {}

            for name, model in models.items():
                outputs = model(images)
                probs = torch.sigmoid(outputs)
                model_preds[name].append(probs.cpu().numpy())
                batch_probs[name] = probs.cpu()

            all_targets.append(masks.numpy())

            # Save first 3 batches for visualization
            if batch_idx < 3:
                batch_data.append((images.cpu(), masks, batch_probs))

    # Flatten everything
    for name in model_preds:
        model_preds[name] = np.concatenate(model_preds[name]).flatten()
    all_targets = np.concatenate(all_targets).flatten()

    return model_preds, all_targets, batch_data


# Run inference
models = {
    'CSNet': csnet,
    'AttentionUNet': attn_unet,
    'SwinUNet': swin_unet,
}

print('Running inference on test set...')
model_preds, all_targets, batch_data = get_all_predictions(models, test_loader, device)
print(f'✅ Done! Collected {len(all_targets):,} pixel predictions from each model.')

## Step 4: Ensemble Strategy 1 — Simple Average (Top 2)

The simplest and often most effective ensemble: average the probability maps from CSNet and Attention U-Net.

```
Ensemble_prob = (CSNet_prob + AttentionUNet_prob) / 2
```

If CSNet predicts 0.7 for a pixel and AttentionUNet predicts 0.6, the ensemble says 0.65 — still a confident vessel prediction.

In [ ]:
# ============================================================
# Cell 9: Ensemble 1 — Simple Average (CSNet + AttentionUNet)
# ============================================================

# Average probabilities from top 2 models
ensemble_2_preds = (model_preds['CSNet'] + model_preds['AttentionUNet']) / 2.0

# Evaluate
results_ensemble_2 = evaluate_full(ensemble_2_preds, all_targets)

print('\n' + '=' * 50)
print('ENSEMBLE 1: Simple Average (CSNet + Attention U-Net)')
print('=' * 50)
for k, v in results_ensemble_2.items():
    print(f'  {k:>14}: {v:.4f}')

## Step 5: Ensemble Strategy 2 — Weighted Average (Top 3)

Now we add Swin-UNet with a smaller weight. This leverages Swin's strength (high sensitivity for thin vessels) without letting its lower specificity hurt the overall score.

```
Ensemble_prob = 0.4 × CSNet + 0.4 × AttentionUNet + 0.2 × SwinUNet
```

The weights must sum to 1.0 so the output stays in the [0, 1] probability range.

In [ ]:
# ============================================================
# Cell 10: Ensemble 2 — Weighted Average (3 models)
# ============================================================

# Weighted combination
W_CSNET = 0.4
W_ATTN = 0.4
W_SWIN = 0.2

ensemble_3_preds = (
    W_CSNET * model_preds['CSNet'] +
    W_ATTN * model_preds['AttentionUNet'] +
    W_SWIN * model_preds['SwinUNet']
)

# Evaluate
results_ensemble_3 = evaluate_full(ensemble_3_preds, all_targets)

print('\n' + '=' * 60)
print(f'ENSEMBLE 2: Weighted (CSNet={W_CSNET}, AttnUNet={W_ATTN}, Swin={W_SWIN})')
print('=' * 60)
for k, v in results_ensemble_3.items():
    print(f'  {k:>14}: {v:.4f}')

## Step 6: Full Comparison Table

Compare all individual models against both ensemble strategies to see the improvement.

In [ ]:
# ============================================================
# Cell 11: Full Comparison Table
# ============================================================

# Evaluate individual models too
results_csnet = evaluate_full(model_preds['CSNet'], all_targets)
results_attn = evaluate_full(model_preds['AttentionUNet'], all_targets)
results_swin = evaluate_full(model_preds['SwinUNet'], all_targets)

all_results = {
    'CSNet (individual)': results_csnet,
    'Attn U-Net (individual)': results_attn,
    'Swin-UNet (individual)': results_swin,
    'Ensemble: Avg(CS+Attn)': results_ensemble_2,
    f'Ensemble: W({W_CSNET}/{W_ATTN}/{W_SWIN})': results_ensemble_3,
}

metrics_list = ['dice', 'iou', 'sensitivity', 'specificity', 'precision', 'auc_roc', 'auc_pr']
header = f"{'Model':<30}" + ''.join([f'{m:>14}' for m in metrics_list])
print('=' * len(header))
print(header)
print('=' * len(header))

for model_name, scores in all_results.items():
    row = f'{model_name:<30}'
    for m in metrics_list:
        val = float(scores.get(m, 0))
        row += f'{val:>14.4f}'
    print(row)

print('=' * len(header))

# Highlight the winner
print('\n★ Best per metric:')
for m in metrics_list:
    best = max(all_results, key=lambda x: float(all_results[x].get(m, 0)))
    val = float(all_results[best].get(m, 0))
    print(f'  {m:>14}: {best} ({val:.4f})')

## Step 7: Visual Comparison

Side-by-side comparison for test images:
- Original Image
- Ground Truth
- CSNet Prediction
- Attention U-Net Prediction
- Swin-UNet Prediction
- Ensemble (Best Strategy) Prediction
- Overlay (Green=GT, Red=Pred, Yellow=Both)

This lets you visually see where the ensemble improves over individual models.

In [ ]:
# ============================================================
# Cell 12: Visual Comparison
# ============================================================

NUM_VIS = 4  # Number of images to visualize
THRESHOLD = 0.5

fig, axes = plt.subplots(NUM_VIS, 7, figsize=(28, 4 * NUM_VIS))
fig.suptitle('Ensemble Comparison — Individual Models vs Combined', fontsize=18, fontweight='bold')

col_titles = ['Original', 'Ground Truth', 'CSNet', 'Attn U-Net', 'Swin-UNet',
              'Ensemble', 'Overlay (Ensemble)']
for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=11, fontweight='bold')

img_count = 0
for images, masks, batch_probs in batch_data:
    for i in range(images.shape[0]):
        if img_count >= NUM_VIS:
            break

        img_np = denormalize(images[i])
        gt_np = masks[i].squeeze().numpy()

        # Individual model predictions
        csnet_prob = batch_probs['CSNet'][i].squeeze().numpy()
        attn_prob = batch_probs['AttentionUNet'][i].squeeze().numpy()
        swin_prob = batch_probs['SwinUNet'][i].squeeze().numpy()

        # Ensemble prediction (use the weighted 3-model version)
        ens_prob = W_CSNET * csnet_prob + W_ATTN * attn_prob + W_SWIN * swin_prob
        ens_bin = (ens_prob > THRESHOLD).astype(np.float32)

        # Plot
        axes[img_count, 0].imshow(img_np)
        axes[img_count, 1].imshow(gt_np, cmap='gray')
        axes[img_count, 2].imshow((csnet_prob > THRESHOLD).astype(float), cmap='gray')
        axes[img_count, 3].imshow((attn_prob > THRESHOLD).astype(float), cmap='gray')
        axes[img_count, 4].imshow((swin_prob > THRESHOLD).astype(float), cmap='gray')
        axes[img_count, 5].imshow(ens_bin, cmap='gray')

        # Overlay
        overlay = img_np.copy()
        overlay[gt_np > 0.5] = [0, 1, 0]       # Green = GT
        overlay[ens_bin > 0.5] = [1, 0, 0]      # Red = Ensemble
        both = (gt_np > 0.5) & (ens_bin > 0.5)
        overlay[both] = [1, 1, 0]               # Yellow = Correct
        axes[img_count, 6].imshow(overlay)

        for j in range(7):
            axes[img_count, j].axis('off')
        img_count += 1

    if img_count >= NUM_VIS:
        break

plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'ensemble_visual_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved: {SAVE_DIR}/ensemble_visual_comparison.png')

In [ ]:
# ============================================================
# Cell 13: ROC & PR Curves — Individual vs Ensemble
# ============================================================
from sklearn.metrics import roc_curve, precision_recall_curve, auc

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Ensemble vs Individual — ROC & PR Curves', fontsize=14, fontweight='bold')

colors = {'CSNet': 'blue', 'AttentionUNet': 'green', 'SwinUNet': 'orange',
          'Ensemble (Avg 2)': 'red', 'Ensemble (W3)': 'purple'}

all_pred_sets = {
    'CSNet': model_preds['CSNet'],
    'AttentionUNet': model_preds['AttentionUNet'],
    'SwinUNet': model_preds['SwinUNet'],
    'Ensemble (Avg 2)': ensemble_2_preds,
    'Ensemble (W3)': ensemble_3_preds,
}

for name, preds in all_pred_sets.items():
    color = colors[name]
    lw = 3 if 'Ensemble' in name else 1.5
    ls = '-' if 'Ensemble' in name else '--'

    # ROC
    fpr, tpr, _ = roc_curve(all_targets, preds)
    roc_auc = auc(fpr, tpr)
    ax1.plot(fpr, tpr, color=color, linewidth=lw, linestyle=ls,
             label=f'{name} (AUC={roc_auc:.4f})')

    # PR
    prec, rec, _ = precision_recall_curve(all_targets, preds)
    pr_auc = auc(rec, prec)
    ax2.plot(rec, prec, color=color, linewidth=lw, linestyle=ls,
             label=f'{name} (AUC={pr_auc:.4f})')

ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve')
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.legend(loc='lower left', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'ensemble_roc_pr_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved: {SAVE_DIR}/ensemble_roc_pr_curves.png')

In [ ]:
# ============================================================
# Cell 14: Bar Chart — Dice Score Comparison
# ============================================================

import os
import matplotlib.pyplot as plt

model_names = list(all_results.keys())
dice_scores = [float(all_results[m]['dice']) for m in model_names]

# Make sure there are enough colors
base_colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6', '#1abc9c', '#34495e']
colors_bar = base_colors[:len(model_names)]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(model_names)), dice_scores, color=colors_bar, edgecolor='black', linewidth=0.5)

# Add value labels on top of bars
for bar, score in zip(bars, dice_scores):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        score + 0.002,
        f'{score:.4f}',
        ha='center',
        va='bottom',
        fontweight='bold',
        fontsize=11
    )

ax.set_xticks(range(len(model_names)))
ax.set_xticklabels(model_names, rotation=25, ha='right', fontsize=10)
ax.set_ylabel('Dice Score', fontsize=12)
ax.set_title('Dice Score: Individual Models vs Ensembles', fontsize=14, fontweight='bold')

# FIX: dynamic y-limits so low scores like Swin-UNet are visible
y_min = min(dice_scores) - 0.01
y_max = max(dice_scores) + 0.02
ax.set_ylim(y_min, y_max)

ax.grid(axis='y', alpha=0.3)
ax.axhline(y=0.80, color='red', linestyle=':', alpha=0.5, label='0.80 target')
ax.legend()

plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'ensemble_dice_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved: {SAVE_DIR}/ensemble_dice_comparison.png')

In [ ]:
# ============================================================
# Cell 15: Save All Results & Download
# ============================================================

# Save JSON results
final_results = {
    'individual_models': {
        'CSNet': results_csnet,
        'AttentionUNet': results_attn,
        'SwinUNet': results_swin,
    },
    'ensemble_avg_2models': results_ensemble_2,
    'ensemble_weighted_3models': results_ensemble_3,
    'weights': {
        'CSNet': W_CSNET,
        'AttentionUNet': W_ATTN,
        'SwinUNet': W_SWIN,
    },
}

with open(os.path.join(SAVE_DIR, 'ensemble_results.json'), 'w') as f:
    json.dump(final_results, f, indent=2, default=str)
print(f'✓ Saved: {SAVE_DIR}/ensemble_results.json')

# Zip and download
!zip -r ensemble_results.zip {SAVE_DIR}/
from google.colab import files
files.download('ensemble_results.zip')
print('\n✅ All done! Downloading ensemble_results.zip')